# Livebuild.ai - Phase 0 spike

**Question this notebook answers:** does monocular depth estimation survive real MLS listing photos?

Everything downstream - the 2.5D shells that give the tour its in-room parallax - depends on the answer.
Listing photos are wide-angle, heavily tone-mapped, and full of blown-out windows, which is a very
different problem from the clean phone captures these models get demoed on.

The dollhouse and the click-to-walk tour do **not** depend on this. They come from the hand-drawn floor
plan. Only the in-room parallax is at stake here, so a bad result narrows the product, it does not kill it.

Runtime: **T4 GPU** (Runtime > Change runtime type > T4). Free tier is enough.


## 1. Install

MoGe-2 is MIT licensed and its ViT-S/B/L checkpoints are cleared for commercial use.


In [ ]:
!pip -q install git+https://github.com/microsoft/MoGe.git
!pip -q install trimesh

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 2. Get the spike script

Either clone the repo, or upload `pipeline/spike.py` by hand with the file browser on the left.


In [ ]:
# Option A - clone your repo (edit the URL)
# !git clone https://github.com/YOURUSER/livebuild-ai.git && cp livebuild-ai/pipeline/spike.py .

# Option B - upload spike.py via the Files pane, then confirm it is here:
import os; print('spike.py present:', os.path.exists('spike.py'))


## 3. Upload one property's photos

All the photos for a single property. 20-30 is a typical MLS set.
Do not mix properties - the point is to judge one house end to end.


In [ ]:
import os, shutil
from google.colab import files

os.makedirs('photos', exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, os.path.join('photos', name))
print(f'{len(os.listdir("photos"))} photos ready')


## 4. Run the spike

Roughly a second or two per photo on a T4.


In [ ]:
!python spike.py --photos photos --out out


## 5. Look at the depth previews

Near = bright. What to look for:
- **Windows** should not be punched out to infinity (black)
- **Walls** should be smooth gradients, not blotchy
- **Mirrors** will be wrong - the question is how wrong
- **Furniture** should separate cleanly from the floor


In [ ]:
import json, glob
import matplotlib.pyplot as plt
from PIL import Image

report = json.load(open('out/report.json'))
worst = sorted(report, key=lambda r: r['tearRisk'], reverse=True)[:6]

fig, axes = plt.subplots(len(worst), 2, figsize=(11, 3.4 * len(worst)))
for ax_row, entry in zip(axes, worst):
    stem = entry['photo'].rsplit('.', 1)[0]
    src = glob.glob(f'photos/{stem}.*')[0]
    ax_row[0].imshow(Image.open(src)); ax_row[0].set_title(entry['photo'], fontsize=9)
    ax_row[1].imshow(Image.open(f'out/depth/{stem}.png'), cmap='magma')
    ax_row[1].set_title(f"tear={entry['tearRisk']:.3f}  budget={entry['parallaxBudget']}m", fontsize=9)
    for a in ax_row: a.axis('off')
plt.suptitle('Highest tear risk first - these are the hardest cases', y=1.0)
plt.tight_layout(); plt.show()


## 6. The actual decision - orbit a point cloud

**This is the gate.** The numbers above only rank which photos to inspect. Judge with your eyes.

Rotate the cloud. A good result looks like a room: flat walls meeting at right angles, furniture
standing proud of the floor. A bad result looks like a crumpled sheet.


In [ ]:
import trimesh, glob

path = sorted(glob.glob('out/cloud/*.ply'))[0]   # or pick a specific one
print('showing', path)
cloud = trimesh.load(path)
print(f'{len(cloud.vertices):,} points')
cloud.show()


## 7. Download everything for a closer look

Open the PLYs in Blender, MeshLab, or https://3dviewer.net for a better look than Colab gives.


In [ ]:
!zip -qr spike-out.zip out
from google.colab import files
files.download('spike-out.zip')


## Decision gate

| What you saw | What to build |
|---|---|
| Walls flat, room shape reads correctly | **Full 2.5D plan.** Feed `report.json` budgets into the schema. |
| Sound but noisy in places | **2.5D with tighter budgets.** Halve `parallaxBudget`. |
| Warped walls, soup, windows to infinity | **Flat photo nodes + dollhouse.** Still ships, still beats a slideshow. |

Record which row you landed on - it decides how Phase 2 gets built.

### If it passed
Copy `out/report.json` next to the property's JSON. The `fovDeg` and `parallaxBudget` values
go straight into its nodes, and the editor will pick them up.

### Worth trying if it was borderline
- MLS photos are often over-sharpened. Try the originals if you can get them.
- Shoot 10 extra plain phone photos in hallways - they also help stitch rooms together.
- Swap the checkpoint: `--model Ruicheng/moge-2-vitb-normal` is faster, ViT-L is more accurate.
